## Count Words

In [1]:
import json
import statistics
from pathlib import Path

MAX_WORDS = 150   # chunk size limit
OVERLAP_WORDS = 30   # words shared between consecutive text chunks
MAX_CONTEXT_WORDS = 40   # keep only the context closest to the table

def n_words(text):
	"""Count words, ignoring markdown table symbols."""
	return sum(1 for w in text.split() if w not in ("|", "---"))

## Chunk Text

In [2]:
def chunk_text(text, max_words=MAX_WORDS, overlap=OVERLAP_WORDS):
	"""Split text into overlapping windows of at most max_words words."""
	words = text.split()
	chunks = []
	start = 0
	while start < len(words):
		end = start + max_words
		chunks.append(" ".join(words[start:end]))
		if end >= len(words):   # reached the end of the text
			break
		start = end - overlap   # step back so chunks share some words
	return chunks

## Chunk Tables

In [3]:
def chunk_table(block, max_words=MAX_WORDS):
	"""Split a table by rows, repeating its context and header in every chunk."""
	lines = block["content"].splitlines()
	header, rows = lines[:2], lines[2:]   # header line + separator, then data rows
	context = " ".join(block["context"].split()[-MAX_CONTEXT_WORDS:])   # last 40 words, nearest the table
	prefix = [context] + header

	chunks = []
	current = []
	for row in rows:
		candidate = current + [row]
		if current and n_words("\n".join(prefix + candidate)) > max_words:
			chunks.append("\n".join(prefix + current))   # close the chunk before it overflows
			current = [row]
		else:
			current = candidate

	if current or not rows:   # last chunk (or a table with no data rows)
		chunks.append("\n".join(prefix + current))
	return chunks

## Chunk a Document

In [4]:
def chunk_document(doc):
	"""Turn a parsed 10-K into chunks with metadata."""
	chunks = []
	# Loops over blocks with their index
	for i, block in enumerate(doc["blocks"]):
		# Uses the method that matches the block type
		if block["type"] == "text":
			pieces = chunk_text(block["content"])
		else:
			pieces = chunk_table(block)

		for j, text in enumerate(pieces):
			chunks.append({
				"id": f"{doc['ticker']}-{i}-{j}",   # e.g. "AAPL-42-1" = block 42, piece 1
				"ticker": doc["ticker"],
				"filing_date": doc["filing_date"],
				"type": block["type"],
				"text": text
			})
	return chunks

## Check on Apple

In [5]:
path = next(Path("../data/processed").glob("AAPL_10-K_*.json"))
doc = json.loads(path.read_text(encoding="utf-8"))
chunks = chunk_document(doc)

sizes = [n_words(c["text"]) for c in chunks]
n_table = sum(c["type"] == "table" for c in chunks)
print(f"{len(chunks)} chunks ({n_table} table, {len(chunks) - n_table} text)")
print(f"words: min {min(sizes)}, median {statistics.median(sizes)}, max {max(sizes)}")

347 chunks (90 table, 257 text)
words: min 3, median 150, max 168


## Inspect a Split Table

In [6]:
# Selects table chunks that are the second or later piece of a split table
split = [c for c in chunks if c["type"] == "table" and not c["id"].endswith("-0")]
print(f"{len(split)} table chunks are continuation pieces\n")
print(split[0]["text"])

31 table chunks are continuation pieces

For the Fiscal Year Ended September 26, 2026 TABLE OF CONTENTS
|  | Page Part I | Page Part I |
| --- | --- | --- |
| Item 10. | Directors, Executive Officers and Corporate Governance | 58 |
| Item 11. | Executive Compensation | 58 |
| Item 12 . | Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters | 58 |
| Item 13 . | Certain Relationships and Related Transactions, and Director Independence | 58 |
| Item 14. | Principal Accountant Fees and Services | 58 |
| Part IV |  |  |
| Item 15. | Exhibit and Financial Statement Schedules | 59 |
| Item 16. | Form 10-K Summary | 62 |


In [7]:
# 1. Largest chunks: why do they exceed the limit?
# Sorts by word count, largest first, and takes the top three.
for c in sorted(chunks, key=lambda c: -n_words(c["text"]))[:3]:
	print(f"{c['id']} ({c['type']}, {n_words(c['text'])} words)")
	print(c["text"][:600])
	print("\n" + "=" * 60 + "\n")

# 2. Tiny chunks: how many, and what are they?
tiny = [c for c in chunks if n_words(c["text"]) < 20]
print(f"{len(tiny)} chunks under 20 words")
for c in tiny[:10]:
	print(c["id"], c["type"], repr(c["text"][:80]))

AAPL-99-0 (table, 168 words)
our opinion on the financial statements, taken as a whole, and we are not, by communicating the critical audit matter below, providing a separate opinion on the critical audit matter or on the account or disclosure to which it relates.
|  | Uncertain Tax Positions |
| --- | --- |
| Description of the Matter | As discussed in Note 8 to the financial statements, the Company is subject to income taxes in the U.S. and numerous foreign jurisdictions. As of September 26, 2026, the total amount of gross unrecognized tax benefits was $22.5 billion, of which $10.6 billion, if recognized, would impact t


AAPL-4-0 (text, 150 words)
Securities registered pursuant to Section 12(g) of the Act: None Indicate by check mark if the Registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act. Yes ☒ No ☐ Indicate by check mark if the Registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Act. Yes ☐ No ☒ Indicate 

In [8]:
"""Finds the chunk holding the income statement's last rows. If the table
was split, this piece should still start with the "Years Ended September 26,
2026" header."""
for c in chunks:
	if "Earnings per share - Basic" in c["text"]:
		print(c["id"], f"({n_words(c['text'])} words)\n")
		print(c["text"])
		break

AAPL-28-0 (146 words)

CONSOLIDATED STATEMENTS OF OPERATIONS (In millions, except number of shares, which are reflected in thousands, and per-share amounts)
|  | Years Ended September 26, 2026 | Years Ended September 27, 2025 | Years Ended September 28, 2024 |
| --- | --- | --- | --- |
| Net sales - Products | 355,533 | 307,003 | 294,866 |
| Net sales - Services | 123,301 | 109,158 | 96,169 |
| Total net sales | 478,834 | 416,161 | 391,035 |
| Cost of sales - Products | 215,874 | 194,116 | 185,233 |
| Cost of sales - Services | 29,358 | 26,844 | 25,119 |
| Total cost of sales | 245,232 | 220,960 | 210,352 |
| Gross margin | 233,602 | 195,201 | 180,683 |
| Operating expenses - Research and development | 45,765 | 34,550 | 31,370 |
| Operating expenses - Selling, general and administrative | 29,774 | 27,601 | 26,097 |
| Total operating expenses | 75,539 | 62,151 | 57,467 |
| Operating income | 158,063 | 133,050 | 123,216 |
| Other income/(expense), net | 1,174 | (321) | 269 |
| Income bef

In [9]:
print(next(c["text"] for c in chunks if c["id"] == "AAPL-28-1"))

CONSOLIDATED STATEMENTS OF OPERATIONS (In millions, except number of shares, which are reflected in thousands, and per-share amounts)
|  | Years Ended September 26, 2026 | Years Ended September 27, 2025 | Years Ended September 28, 2024 |
| --- | --- | --- | --- |
| Earnings per share - Diluted | 8.98 | 7.46 | 6.08 |
| Shares used in computing earnings per share - Basic | 14,661,935 | 14,948,500 | 15,343,783 |
| Shares used in computing earnings per share - Diluted | 14,720,306 | 15,004,697 | 15,408,095 |


In [11]:
from collections import Counter

with open("../data/chunks.jsonl", encoding="utf-8") as f:
	all_chunks = [json.loads(line) for line in f]   # one chunk per line

big = [c for c in all_chunks if n_words(c["text"]) > 200]
tiny = [c for c in all_chunks if n_words(c["text"]) < 5]
print(f"{len(big)} chunks over 200 words, {len(tiny)} chunks under 5 words\n")
# Counts oversized chunks per company, to see if one filing format causes them
print("Over 200 by company:", Counter(c["ticker"] for c in big).most_common(), "\n")

for c in sorted(big, key=lambda c: -n_words(c["text"]))[:5]:
	lines = c["text"].splitlines()
	prefix_words = n_words("\n".join(lines[:3]))   # context + header + separator
	longest_row = max((n_words(line) for line in lines[3:]), default=0)   # biggest data row
	print(f"{c['id']} ({c['type']}, {n_words(c['text'])} words) "
	      f"prefix={prefix_words}, longest row={longest_row}")
	print(c["text"][:300])
	print()

print("Tiny examples:", [c["text"] for c in tiny[:10]])


0 chunks over 200 words, 35 chunks under 5 words

Over 200 by company: [] 

Tiny examples: ['Other Non-Current Assets', 'Other Current Liabilities', 'Actuarial Assumptions', '(Continued)', '5. Other income (expense)', 'U.S. Upstream', 'Financial Table of Contents', 'Financial Table of Contents', 'Financial Table of Contents', 'Financial Table of Contents']
